# Format 2 — Hierarchical / System 1 + System 2
## LLM plans. RL executes.

**Gemini's prediction: 25% probability. Strategically the one Meta would MOST want you to win with.**

The core idea comes from cognitive science (Kahneman) and neuroscience:
- **System 2** — slow, deliberate reasoning. Great for planning. Bad at real-time reactions.
- **System 1** — fast, reflexive control. Great at real-time. Terrible at long-horizon.

In our stack:
- **LLM (Llama)** = System 2 — handles multi-day reasoning, reads forecasts, decides strategy.
- **RL (PPO)** = System 1 — executes every step in <1 ms, reactive control.

**This notebook's demo is the one that will most impress Meta judges.** It's the "Llama + PyTorch RL working together" narrative Gemini said judges will eat up.

### What you'll learn
1. **Why flat RL fails** on long-horizon tasks
2. **Options framework** (hierarchical RL, minimally)
3. **Goal-conditioned policies** — the RL side
4. **LLM as high-level planner** — the LLM side
5. **Wiring them together** on GridOps Task 2 (heatwave — multi-day planning REQUIRED)
6. **Neuro-symbolic demo narrative**

---
## Part 1 — Why flat RL fails on long horizons

### The credit assignment problem

Imagine the agent takes action 0 (charge battery) at step 0 (night, Day 1). Blackout avoided at step 40 (Day 2 evening). The reward signal for that smart decision arrives **40 steps later**.

Flat PPO's advantage estimator must propagate that signal through 40 time steps of noisy intermediate rewards. With γ=0.99, the signal is multiplied by 0.99⁴⁰ ≈ 0.67 — still alive. With γ=0.95, it becomes 0.95⁴⁰ ≈ 0.13 — nearly lost.

```
decision at t=0  ──────────────── 40 steps of noise ────────────────►  reward at t=40
                                       ↑
                            signal gets diluted here
```

### Why humans don't have this problem

Humans **decompose**. We don't think "what's my action for hour 17?" We think: "today's plan is: conserve battery for Day 2 peak." Then, within that plan, we pick hour-by-hour moves.

The plan *is the prior* for the local moves. That's hierarchy.

### The fix: two policies at two time scales

| Policy | Frequency | Input | Output |
|---|---|---|---|
| **High (LLM)** | every 24 steps | current state + forecasts | **macro-goal** (text or structured) |
| **Low (RL)** | every step | current obs + macro-goal | action |

Credit assignment window shrinks from 72 → 24 for low-level, and high-level only makes 3 decisions per episode. Both become trainable.

---
## Part 2 — Options framework (intuition, not math)

The options framework (Sutton, Precup, Singh 1999) is the OG formalism for HRL. Core pieces:

1. **Option**: a temporally extended action = a sub-policy + termination condition.
   Example option: `"conserve_battery"` = "charge aggressively until SOC > 80% OR 12 steps pass."

2. **Meta-policy**: chooses which option to execute next.

3. **Intra-option policy**: executes the chosen option step-by-step.

In our LLM+RL setup:
- Options = the macro-goals Llama can issue (`conserve`, `arbitrage_aggressive`, `survival_mode`, ...)
- Meta-policy = Llama reading the forecasts
- Intra-option policy = PPO conditioned on the chosen option

You don't need to *implement* options formally. You just need **two levels**, which is what we'll build.

---
## Part 3 — Goal-conditioned policies

The RL side of the sandwich. A normal PPO policy maps `obs → action`. A goal-conditioned one maps `(obs, goal) → action`.

```
Normal:            obs  ──►  π(·)  ──►  action
Goal-conditioned: (obs, goal) ──►  π(·)  ──►  action
```

**Concretely for GridOps:** we augment the observation vector with a one-hot of the current macro-goal. The same network weights handle all goals; the one-hot flips which mode the policy should operate in.

Why this works: with enough training, the policy learns "when the goal slot says CONSERVE, I should charge; when it says ARBITRAGE, I should discharge." It's one policy with multiple behavioral modes.

In [ ]:
import sys, os, json
import numpy as np
sys.path.insert(0, os.path.abspath('.'))

from gridops.server.environment import GridOpsEnvironment
from gridops.models import GridOpsAction

# ─── Four macro-goals the LLM can issue ───
GOALS = ['CONSERVE', 'ARBITRAGE', 'SURVIVAL', 'NORMAL']
GOAL_TO_IDX = {g: i for i, g in enumerate(GOALS)}

def goal_onehot(goal: str) -> np.ndarray:
    v = np.zeros(len(GOALS), dtype=np.float32)
    v[GOAL_TO_IDX[goal]] = 1.0
    return v

def flatten_obs_with_goal(obs, goal: str) -> np.ndarray:
    base = np.array([
        obs.hour / 72.0, obs.demand_kw / 500.0, obs.solar_kw / 250.0,
        obs.battery_soc, obs.grid_price / 20.0, obs.diesel_fuel_remaining,
        float(obs.diesel_is_on), obs.day_of_episode / 3.0,
    ], dtype=np.float32)
    forecasts = []
    for lst, scale in [(obs.demand_forecast_4h, 500.0), (obs.solar_forecast_4h, 250.0), (obs.price_forecast_4h, 20.0)]:
        padded = (list(lst) + [0.0]*4)[:4]
        forecasts.extend([x/scale for x in padded])
    return np.concatenate([base, np.array(forecasts, dtype=np.float32), goal_onehot(goal)])

env = GridOpsEnvironment()
obs = env.reset(seed=42, task_id='task_2_heatwave')
v = flatten_obs_with_goal(obs, 'CONSERVE')
print(f'Augmented obs shape: {v.shape} (20 state dims + 4 goal dims = {20+len(GOALS)})')
print(f'Goal one-hot (CONSERVE): {v[-4:]}')

---
## Part 4 — The LLM planner

The planner runs **every 24 steps** (once per day). It sees:
- Current state (SOC, fuel, cumulative blackout)
- Forecasts for next 4 hours
- Day of episode (are we entering a heatwave?)

It outputs one of the 4 goals as a string.

### Why once per day?

Decision frequency matters. Too slow → can't react to sudden events. Too fast → LLM calls blow your latency + token budget. Once per day is the natural cadence for a microgrid (you wake up, decide the day's strategy, execute).

For the hackathon, you might make it **event-driven** instead: re-plan only when something unexpected happens (cloud cover dropped solar 50%, price spiked). That's more sophisticated and shows judges you're thinking about inference cost.

In [ ]:
from openai import OpenAI
client = OpenAI(
    base_url=os.getenv('API_BASE_URL', 'https://router.huggingface.co/v1'),
    api_key=os.getenv('HF_TOKEN', 'dummy'),
)
MODEL = os.getenv('MODEL_NAME', 'meta-llama/Llama-3.3-70B-Instruct')

PLANNER_SYSTEM = """You are a microgrid PLANNER. Once per day you set the strategy.
Your output must be ONE of: CONSERVE, ARBITRAGE, SURVIVAL, NORMAL.

Meanings:
- CONSERVE: keep battery high for upcoming crisis (heatwave, grid outage, price spike)
- ARBITRAGE: aggressively buy-low / sell-high — safe conditions
- SURVIVAL: crisis mode. Ration diesel, allow shedding, maintain reliability at all cost
- NORMAL: balanced operation

Reply with the single word, no explanation."""

def llm_planner(obs, day: int) -> str:
    """Called once per day. Reads forecasts + current state, outputs a goal."""
    prompt = (
        f'Day {day}. SOC={obs.battery_soc*100:.0f}%, diesel_fuel={obs.diesel_fuel_remaining*100:.0f}%\n'
        f'Blackouts so far: {obs.cumulative_blackout_kwh:.1f} kWh, cost: Rs {obs.cumulative_cost:.0f}\n'
        f'Price forecast: {[f"{p:.1f}" for p in obs.price_forecast_4h]}\n'
        f'Demand forecast: {[f"{d:.0f}" for d in obs.demand_forecast_4h]}\n'
        f'Solar forecast: {[f"{s:.0f}" for s in obs.solar_forecast_4h]}\n'
        f'Your goal for today? (CONSERVE/ARBITRAGE/SURVIVAL/NORMAL)'
    )
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'system', 'content': PLANNER_SYSTEM}, {'role': 'user', 'content': prompt}],
            temperature=0.0, max_tokens=20,
        )
        text = resp.choices[0].message.content.strip().upper()
        for g in GOALS:
            if g in text: return g
    except Exception:
        pass
    return 'NORMAL'

# Rule-based fallback (no API needed) — deterministic, useful for demo without LLM
def rule_planner(obs, day: int) -> str:
    if obs.cumulative_blackout_kwh > 5: return 'SURVIVAL'
    avg_price_fc = np.mean(obs.price_forecast_4h) if obs.price_forecast_4h else 8.0
    avg_dem_fc = np.mean(obs.demand_forecast_4h) if obs.demand_forecast_4h else 200.0
    if avg_dem_fc > 250 and day >= 2: return 'CONSERVE'
    if avg_price_fc > 12: return 'ARBITRAGE'
    return 'NORMAL'

print('Two planners defined: llm_planner (real LLM) and rule_planner (for testing without API).')

---
## Part 5 — The RL executor

For this notebook we use a **goal-conditioned heuristic** as the executor instead of a trained PPO network. Why? Because:
1. The goal-conditioned PPO would take minutes to train — too slow for illustration.
2. A **hand-coded** executor makes it easier to see that the *goal* is doing the work, not the executor.
3. In the real hackathon, you'd swap this heuristic for a trained PPO whose obs vector includes the goal one-hot (Part 3).

Each goal has a different rule book. Same observation, different action depending on the goal.

In [ ]:
def goal_conditioned_executor(obs, goal: str) -> GridOpsAction:
    step_in_day = int(obs.hour) % 24  # 0 = 6 AM, 12 = 6 PM, 18 = midnight
    soc = obs.battery_soc; demand = obs.demand_kw; solar = obs.solar_kw; price = obs.grid_price
    bat, diesel, shed = 0.0, 0.0, 0.0

    if goal == 'CONSERVE':
        # Only discharge when there would otherwise be blackout
        if 18 <= step_in_day <= 23 and soc < 0.95: bat = -1.0
        elif 12 <= step_in_day <= 17:
            gap = demand - solar - 200
            if gap > 0 and soc > 0.30: bat = float(np.clip(gap / 100, 0, 1))

    elif goal == 'ARBITRAGE':
        # Aggressive: charge at cheap hours, discharge when price > 10
        if 18 <= step_in_day <= 23 and soc < 0.95: bat = -1.0
        elif price > 10 and soc > 0.3:
            # Export! Discharge even if no gap
            bat = 0.7
        elif 12 <= step_in_day <= 17:
            gap = demand - solar - 200
            if gap > 0: bat = float(np.clip(gap / 100, 0, 1))

    elif goal == 'SURVIVAL':
        # Keep lights on at all cost
        if 18 <= step_in_day <= 23 and soc < 1.0: bat = -1.0
        elif 12 <= step_in_day <= 17:
            gap = demand - solar - 200
            if gap > 0:
                bat = float(np.clip(gap / 100, 0, 1))
                if soc < 0.20: diesel = 0.9
                if soc < 0.10: shed = 0.5

    else:  # NORMAL
        if 18 <= step_in_day <= 23 and soc < 0.90: bat = -0.7
        elif 12 <= step_in_day <= 17:
            gap = demand - solar - 200
            if gap > 0 and soc > 0.15: bat = float(np.clip(gap / 100, 0, 1))

    return GridOpsAction(
        battery_dispatch=float(np.clip(bat, -1, 1)),
        diesel_dispatch=float(np.clip(diesel, 0, 1)),
        demand_shedding=float(np.clip(shed, 0, 1)),
    )

print('Goal-conditioned executor defined. Same obs → 4 different actions depending on goal.')

---
## Part 6 — Wiring it all together

The hierarchical loop:

```
for step in range(72):
    if step % 24 == 0:  # once per day
        goal = llm_planner(obs, day=step // 24 + 1)  # LLM decides strategy
    action = executor(obs, goal)                       # RL / heuristic executes
    obs = env.step(action)
```

That's System 1 / System 2 in <10 lines.

Below: compare **flat** (goal='NORMAL' always) vs **hierarchical** (goal chosen daily) on Task 2 (heatwave). Task 2 is where multi-day planning actually matters — a flat NORMAL policy will underperform because it doesn't know Day 2 is worse than Day 1.

In [ ]:
def run_hierarchical(env, planner, task_id='task_2_heatwave', seed=42, use_hierarchy=True):
    obs = env.reset(seed=seed, task_id=task_id)
    total_r, blackout = 0.0, 0.0
    goal_log = []
    current_goal = 'NORMAL'
    for step in range(72):
        if use_hierarchy and step % 24 == 0:
            current_goal = planner(obs, day=step // 24 + 1)
            goal_log.append((step, current_goal))
        action = goal_conditioned_executor(obs, current_goal)
        obs = env.step(action)
        total_r += obs.reward or 0.0
        blackout += obs.flow_blackout
        if obs.done: break
    grade = env.state.grade
    return {'reward': total_r, 'blackout': blackout, 'grade': grade, 'goals': goal_log}

# Use rule_planner here so it runs without API key. Swap to llm_planner when testing with Llama.
print('── Flat (no hierarchy, always NORMAL) ──')
flat = run_hierarchical(env, planner=rule_planner, use_hierarchy=False)
print(f'  reward={flat["reward"]:.2f}  blackout={flat["blackout"]:.1f} kWh  grade={flat["grade"]}')

print('\n── Hierarchical (planner chooses daily goal) ──')
hier = run_hierarchical(env, planner=rule_planner, use_hierarchy=True)
print(f'  reward={hier["reward"]:.2f}  blackout={hier["blackout"]:.1f} kWh  grade={hier["grade"]}')
print(f'  daily goals chosen: {hier["goals"]}')

---
## Part 7 — The neuro-symbolic narrative

Here's the pitch for Meta's judges, exactly as Gemini suggested:

> "LLMs reason brilliantly over long horizons but can't output 72 actions per second. RL executes sub-millisecond but loses the plot on multi-day problems. **We combined them using OpenEnv.** Llama reads the forecast and sets a macro-goal. PPO conditions on the goal and executes step-by-step. The result: reliability > 99% on heatwave scenarios where each component alone fails."

### Why this beats "RL > LLM"

| Narrative | What Meta hears |
|---|---|
| *"PPO outperforms Llama on this benchmark"* | "Your billion-dollar model lost to a 2017 algorithm." |
| **"PPO + Llama together beat either alone"** | **"Your model plus your RL framework = something new."** |

The first makes them wince. The second makes them invest.

### The demo that lands

Show **three runs** live on stage, side by side:
1. LLM only — reasonable strategy, but slow and inconsistent
2. PPO only — fast but blind to the heatwave
3. **LLM + PPO hierarchy** — both advantages, no drawbacks

Plot the three learning/reliability curves. The hierarchy line sits cleanly above both baselines. End on it.

---

## Extensions worth building in 48 hours

If you commit to this format:

1. **Event-driven re-planning** — instead of every 24 steps, re-plan when state deviates from forecast by > X%. Shows you're thinking about inference budget.
2. **Chain-of-thought goal choice** — ask Llama to *explain* why it picked SURVIVAL. Display the reasoning in your dashboard.
3. **Goal-conditioned PPO (the real version)** — replace the heuristic executor with a network trained on (obs, goal_onehot) pairs. Goal slot becomes a learned embedding.
4. **Goal ablation** — show what happens with the wrong goal. "CONSERVE during normal day → lower profit but safe. ARBITRAGE during heatwave → catastrophic blackout." Proves goals matter.

Each of these is 2-4 hours of work and adds visible differentiation.